# 04 — Estrazione delle feature prosodiche su train e dev completi

Estrae le 8 feature per finestra (200 ms) su **tutti** i file di train (25.380) e dev (24.844) di ASVspoof 2019 LA,
con due configurazioni Praat:
- `standard`: valori standard di Praat;
- `rand18`: la migliore della ricerca casuale (notebook 03).

L'output è il **dataset definitivo** della prosodia.

**Formato di output** (`/kaggle/working/prosody_features/`), per ogni configurazione e split:
`{cfg}_{split}.npz` con `X` (tutte le finestre concatenate, float32, 8 colonne), `offsets` (inizio di ogni file in `X`,
lunghezza N+1) e `utt_id`. Il file `index.csv` contiene etichette e statistiche per file.

In [1]:
import os, sys, re, json, time, shutil, subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

try:
    import parselmouth
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "praat-parselmouth"], check=True)
    import parselmouth
from parselmouth.praat import call

INPUT_ROOT = Path(os.environ.get("INPUT_ROOT", "/kaggle/input"))
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/prosody_features"))
PART_DIR = OUT_DIR / "_parts"
PART_DIR.mkdir(parents=True, exist_ok=True)
N_WORKERS = os.cpu_count() or 1
CHUNK = int(os.environ.get("CHUNK", 1000))
DELETE_PARTS = True          # a fine lavoro elimina i blocchi intermedi

# Parametri fissi (identici ai notebook 02 e 03)
PITCH_FLOOR, PITCH_CEILING = 75.0, 500.0
MAX_CANDIDATES, VERY_ACCURATE, VOICING_THRESHOLD, TIME_STEP = 15, "no", 0.45, 0.0
PERIOD_FLOOR, PERIOD_CEILING = 0.8 / PITCH_CEILING, 1.25 / PITCH_FLOOR
MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR = 1.3, 1.6
HNR_TIME_STEP, HNR_SILENCE, HNR_PERIODS = 0.01, 0.1, 1.0
WINDOW_S = 0.2

CONFIGS = {
    "standard": {"silence_threshold": 0.03,  "octave_cost": 0.01,  "octave_jump_cost": 0.35,  "voiced_unvoiced_cost": 0.14},
    "rand18":   {"silence_threshold": 0.172, "octave_cost": 0.047, "octave_jump_cost": 0.058, "voiced_unvoiced_cost": 0.353},
}
FEATURES = ["f0_mean", "f0_std", "jitter_local", "shimmer_local",
            "hnr_mean", "hnr_std", "intensity_mean", "intensity_std"]
EXPECTED_ROWS = {"train": 25380, "dev": 24844}
print(f"parselmouth {parselmouth.__version__} | Praat {parselmouth.PRAAT_VERSION} | core {N_WORKERS}")

parselmouth 0.4.7 | Praat 6.1.38 | core 4


In [2]:
def find_la_root(root):
    for dirpath, dirnames, _ in os.walk(root):
        dirnames[:] = [d for d in dirnames if d != "flac"]
        if "ASVspoof2019_LA_cm_protocols" in dirnames:
            return Path(dirpath)
    raise RuntimeError("Cartella con ASVspoof2019_LA_cm_protocols non trovata.")

LA_ROOT = find_la_root(INPUT_ROOT)
PATTERNS = {"train": r"cm[._]train[._]trn", "dev": r"cm[._]dev[._]trl"}
frames = []
for split, pat in PATTERNS.items():
    f = [p for p in (LA_ROOT / "ASVspoof2019_LA_cm_protocols").iterdir() if re.search(pat, p.name)]
    assert len(f) == 1, f
    df = pd.read_csv(f[0], sep=r"\s+", header=None, dtype=str, keep_default_na=False, engine="python",
                     names=["speaker", "utt_id", "unused", "attack", "label"])
    df["split"] = split
    df["path"] = [str(LA_ROOT / f"ASVspoof2019_LA_{split}" / "flac" / f"{u}.flac") for u in df["utt_id"]]
    exp = EXPECTED_ROWS[split]
    print(f"{split}: {len(df)} righe" + ("" if len(df) == exp else f"  <-- ATTENZIONE, attese {exp}"))
    frames.append(df)
files = pd.concat(frames, ignore_index=True).drop(columns="unused")
files["attack"] = files["attack"].replace("-", "bonafide")
files["y_bonafide"] = (files["label"] == "bonafide").astype(np.int8)
missing = [p for p in files["path"] if not os.path.exists(p)]
print("File mancanti:", len(missing))
assert not missing, missing[:5]

train: 25380 righe
dev: 24844 righe
File mancanti: 0


In [3]:
def make_windows(duration, w):
    starts = np.arange(0.0, duration, w)
    ends = np.minimum(starts + w, duration)
    keep = (ends - starts) >= w / 2
    return starts[keep], ends[keep]

def window_features(snd, pp, ht, hv, it, iv, a, b, pt, f0):
    f0w = f0[(pt >= a) & (pt < b)]
    f0w = f0w[f0w > 0]
    if len(f0w) == 0:
        return np.zeros(len(FEATURES))
    out = np.full(len(FEATURES), np.nan)
    out[0] = f0w.mean()
    out[1] = f0w.std() if len(f0w) >= 2 else np.nan
    try:
        out[2] = call(pp, "Get jitter (local)", a, b, PERIOD_FLOOR, PERIOD_CEILING, MAX_PERIOD_FACTOR)
    except Exception:
        pass
    try:
        out[3] = call([snd, pp], "Get shimmer (local)", a, b, PERIOD_FLOOR, PERIOD_CEILING,
                      MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR)
    except Exception:
        pass
    hw = hv[(ht >= a) & (ht < b)]; hw = hw[hw > -100]
    if len(hw):
        out[4] = hw.mean(); out[5] = hw.std() if len(hw) >= 2 else np.nan
    iw = iv[(it >= a) & (it < b)]; iw = iw[np.isfinite(iw)]
    if len(iw):
        out[6] = iw.mean(); out[7] = iw.std() if len(iw) >= 2 else np.nan
    return np.nan_to_num(out, nan=0.0)

def process_file(path):
    # {cfg: (X, voiced_frame_frac, n_nan_windows)} per un file; errori restituiti come stringa
    try:
        snd = parselmouth.Sound(path)
        harm = call(snd, "To Harmonicity (cc)", HNR_TIME_STEP, PITCH_FLOOR, HNR_SILENCE, HNR_PERIODS)
        inten = snd.to_intensity(minimum_pitch=PITCH_FLOOR)
        ht, hv = np.asarray(harm.xs()), np.asarray(harm.values[0])
        it, iv = np.asarray(inten.xs()), np.asarray(inten.values[0])
        starts, ends = make_windows(snd.duration, WINDOW_S)
        out = {}
        for name, cfg in CONFIGS.items():
            pitch = call(snd, "To Pitch (cc)", TIME_STEP, PITCH_FLOOR, MAX_CANDIDATES, VERY_ACCURATE,
                         cfg["silence_threshold"], VOICING_THRESHOLD, cfg["octave_cost"],
                         cfg["octave_jump_cost"], cfg["voiced_unvoiced_cost"], PITCH_CEILING)
            pp = call([snd, pitch], "To PointProcess (cc)")
            pt, f0 = np.asarray(pitch.xs()), np.asarray(pitch.selected_array["frequency"])
            X = np.zeros((max(len(starts), 1), len(FEATURES)), dtype=np.float32)
            for i, (a, b) in enumerate(zip(starts, ends)):
                X[i] = window_features(snd, pp, ht, hv, it, iv, a, b, pt, f0)
            out[name] = (X, float((f0 > 0).mean()) if len(f0) else 0.0)
        return out, snd.duration, ""
    except Exception as e:
        return None, np.nan, repr(e)

In [4]:
def part_path(split, k):
    return PART_DIR / f"{split}_{k:05d}.npz"

t_start = time.perf_counter()
for split in ["train", "dev"]:
    sub = files[files["split"] == split].reset_index(drop=True)
    n_chunks = (len(sub) + CHUNK - 1) // CHUNK
    for k in range(n_chunks):
        if part_path(split, k).exists():
            continue
        chunk = sub.iloc[k * CHUNK:(k + 1) * CHUNK]
        t0 = time.perf_counter()
        with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
            res = list(ex.map(process_file, chunk["path"], chunksize=8))
        payload = {"utt_id": chunk["utt_id"].to_numpy(), "duration": np.array([r[1] for r in res]),
                   "error": np.array([r[2] for r in res])}
        for name in CONFIGS:
            arr = np.empty(len(res), dtype=object)
            arr[:] = [r[0][name][0] if r[0] else np.zeros((1, len(FEATURES)), np.float32) for r in res]
            payload[f"X_{name}"] = arr
            payload[f"vf_{name}"] = np.array([r[0][name][1] if r[0] else np.nan for r in res])
        np.savez(part_path(split, k), **payload)
        dt = time.perf_counter() - t0
        print(f"{split} blocco {k + 1}/{n_chunks}: {dt:.0f} s  "
              f"(stima residua per questo split: ~{dt * (n_chunks - k - 1) / 60:.0f} min)")
print(f"Estrazione terminata in {(time.perf_counter() - t_start) / 60:.1f} min")

train blocco 1/26: 99 s  (stima residua per questo split: ~41 min)
train blocco 2/26: 91 s  (stima residua per questo split: ~36 min)
train blocco 3/26: 96 s  (stima residua per questo split: ~37 min)
train blocco 4/26: 90 s  (stima residua per questo split: ~33 min)
train blocco 5/26: 91 s  (stima residua per questo split: ~32 min)
train blocco 6/26: 71 s  (stima residua per questo split: ~24 min)
train blocco 7/26: 109 s  (stima residua per questo split: ~34 min)
train blocco 8/26: 119 s  (stima residua per questo split: ~36 min)
train blocco 9/26: 114 s  (stima residua per questo split: ~32 min)
train blocco 10/26: 104 s  (stima residua per questo split: ~28 min)
train blocco 11/26: 108 s  (stima residua per questo split: ~27 min)
train blocco 12/26: 101 s  (stima residua per questo split: ~24 min)
train blocco 13/26: 92 s  (stima residua per questo split: ~20 min)
train blocco 14/26: 95 s  (stima residua per questo split: ~19 min)
train blocco 15/26: 95 s  (stima residua per questo

In [5]:
index_parts = []
for split in ["train", "dev"]:
    parts = sorted(PART_DIR.glob(f"{split}_*.npz"))
    zs = [np.load(p, allow_pickle=True) for p in parts]
    utt = np.concatenate([z["utt_id"] for z in zs])
    info = pd.DataFrame({"utt_id": utt, "duration": np.concatenate([z["duration"] for z in zs]),
                         "error": np.concatenate([z["error"] for z in zs])})
    for name in CONFIGS:
        seqs = [x for z in zs for x in z[f"X_{name}"]]
        lens = np.array([len(x) for x in seqs])
        offsets = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)
        X = np.concatenate(seqs).astype(np.float32)
        np.savez(OUT_DIR / f"{name}_{split}.npz", X=X, offsets=offsets, utt_id=utt)
        info[f"n_windows"] = lens
        info[f"voiced_frac_{name}"] = np.concatenate([z[f"vf_{name}"] for z in zs])
        info[f"zero_window_frac_{name}"] = [float((np.abs(x).sum(1) == 0).mean()) for x in seqs]
        print(f"{name}_{split}.npz: {len(utt)} file, X {X.shape}, {X.nbytes / 1e6:.1f} MB")
    index_parts.append(info)

index = files.drop(columns="path").merge(pd.concat(index_parts), on="utt_id", how="left", validate="one_to_one")
index.to_csv(OUT_DIR / "index.csv", index=False)

n_err = (index["error"].fillna("") != "").sum()
print("\nFile con errore:", n_err)
if n_err:
    display(index.loc[index["error"].fillna("") != "", ["utt_id", "split", "error"]].head())
for split, exp in EXPECTED_ROWS.items():
    n = (index["split"] == split).sum()
    print(f"{split}: {n} file elaborati (attesi {exp})")
exp_w = np.maximum(np.floor(index["duration"] / WINDOW_S + 0.5), 1)
print("n_windows coerente con la durata:", bool((index["n_windows"] == exp_w).all()))

standard_train.npz: 25380 file, X (434766, 8), 13.9 MB
rand18_train.npz: 25380 file, X (434766, 8), 13.9 MB
standard_dev.npz: 24844 file, X (432089, 8), 13.8 MB
rand18_dev.npz: 24844 file, X (432089, 8), 13.8 MB

File con errore: 0
train: 25380 file elaborati (attesi 25380)
dev: 24844 file elaborati (attesi 24844)
n_windows coerente con la durata: False


In [6]:
cols = [c for c in index.columns if c.startswith(("voiced_frac_", "zero_window_frac_"))] + ["n_windows", "duration"]
summary = index.groupby(["split", "label"])[cols].mean().round(3)
summary.to_csv(OUT_DIR / "summary_by_class.csv")
display(summary)

meta = {"configs": CONFIGS, "window_s": WINDOW_S, "features": FEATURES,
        "silence_rule": "finestra senza frame sonori -> tutte le feature a 0; misure indefinite -> 0",
        "fixed_params": {"pitch_floor": PITCH_FLOOR, "pitch_ceiling": PITCH_CEILING,
                         "max_candidates": MAX_CANDIDATES, "very_accurate": VERY_ACCURATE,
                         "voicing_threshold": VOICING_THRESHOLD, "time_step": TIME_STEP,
                         "period_floor": PERIOD_FLOOR, "period_ceiling": PERIOD_CEILING,
                         "max_period_factor": MAX_PERIOD_FACTOR, "max_amplitude_factor": MAX_AMPLITUDE_FACTOR,
                         "hnr": [HNR_TIME_STEP, PITCH_FLOOR, HNR_SILENCE, HNR_PERIODS],
                         "intensity_minimum_pitch": PITCH_FLOOR},
        "praat_version": parselmouth.PRAAT_VERSION, "parselmouth_version": parselmouth.__version__,
        "n_files": {s: int((index["split"] == s).sum()) for s in EXPECTED_ROWS}, "n_errors": int(n_err),
        "format": "npz per cfg/split: X (finestre concatenate, float32), offsets (N+1), utt_id"}
(OUT_DIR / "meta.json").write_text(json.dumps(meta, indent=2))

if DELETE_PARTS and n_err == 0:
    shutil.rmtree(PART_DIR)
print("Output:", sorted(p.name for p in OUT_DIR.iterdir()))

voiced_frac_standard  zero_window_frac_standard  \
split label                                                       
dev   bonafide                 0.339                      0.454   
      spoof                    0.452                      0.299   
train bonafide                 0.358                      0.426   
      spoof                    0.453                      0.298   

                voiced_frac_rand18  zero_window_frac_rand18  n_windows  \
split label                                                              
dev   bonafide               0.270                    0.511     17.544   
      spoof                  0.359                    0.353     17.375   
train bonafide               0.280                    0.487     16.940   
      spoof                  0.358                    0.353     17.152   

                duration  
split label               
dev   bonafide     3.508  
      spoof        3.475  
train bonafide     3.389  
      spoof        3.430

Output: ['index.csv', 'meta.json', 'rand18_dev.npz', 'rand18_train.npz', 'standard_dev.npz', 'standard_train.npz', 'summary_by_class.csv']
